# Validación y métrica de negocio: M5 Forecast Pipeline

Evalúa el modelo final de la sesión pasada (**Random Forest sin regularizar**) y el segundo mejor (**Red neuronal**) con validación cruzada, con dos esquemas:

- **Temporal:** Cada pliegue entrena con días anteriores y valida con los 4 días siguientes.
- **Al azar:** `KFold` con las filas barajadas.

La validación cruzada se hace **solo dentro del conjunto de entrenamiento**, la semana de prueba no se toca.

In [1]:
import os
import time
import gc
import warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from pipeline_proyecto import NaiveEstacional, cargar_datos, dividir, nueva_preparacion
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [2]:

datos = cargar_datos()
X_ent, X_prueba, y_ent, y_prueba, division = dividir(datos)
fechas_ent = datos.loc[X_ent.index, "DATE"]

print(f"Entrenamiento (donde se hace la validacion cruzada): {division['entrenamiento_desde']} "
      f"a {division['entrenamiento_hasta']}, {len(X_ent):,} filas, {fechas_ent.nunique()} dias")
del datos, X_prueba, y_prueba       # liberar memoria

Entrenamiento (donde se hace la validacion cruzada): 2016-04-20 a 2016-05-15, 792,740 filas, 26 dias


## 1. Métricas

**WAPE** = Σ|error| / Σ unidades reales: la métrica que se venía usando. Trata igual sobreestimar y subestimar.

**Costo asimétrico (métrica de negocio):** El pronóstico alimenta el stock de seguridad, y los dos errores no cuestan lo mismo:

- **Subestimar:** Falta producto en la góndola, se pierde la venta y su margen, y el cliente puede irse a otra tienda.
- **Sobreestimar:** Sobra inventario, cuesta mantenerlo (capital inmovilizado, espacio) y, en comida, puede vencerse.

`costo relativo = Σ precio × (1,0 × faltante + 0,2 × sobrante) / Σ precio × unidades reales`

In [3]:
COSTO_FALTANTE = 1.0   # por unidad no cubierta, como fraccion del precio
COSTO_SOBRANTE = 0.2   # por unidad que sobra, como fraccion del precio


def wape(real, pred):
    real, pred = np.asarray(real, float), np.asarray(pred, float)
    return np.abs(real - pred).sum() / real.sum()


def costo_relativo(real, pred, precio):
    real, pred, precio = (np.asarray(v, float) for v in (real, pred, precio))
    faltante = np.clip(real - pred, 0, None)
    sobrante = np.clip(pred - real, 0, None)
    costo = (precio * (COSTO_FALTANTE * faltante + COSTO_SOBRANTE * sobrante)).sum()
    return costo / (precio * real).sum()


def sesgo(real, pred):
    real, pred = np.asarray(real, float), np.asarray(pred, float)
    return (pred.sum() - real.sum()) / real.sum()

## 2. Los dos esquemas de validación

- **Temporal:** `TimeSeriesSplit` sobre los 26 días de entrenamiento, con 4 pliegues de 4 días.
- **Al azar:** `KFold(n_splits=4, shuffle=True, random_state=42)` sobre las mismas filas. Mezcla días: entrena con el futuro de lo que valida.

In [4]:
N_PLIEGUES = 3
dias = np.sort(fechas_ent.unique())

esquemas = []   # (esquema, pliegue, posiciones de entrenamiento, posiciones de validacion)

tss = TimeSeriesSplit(n_splits=N_PLIEGUES, test_size=4)
for k, (d_ent, d_val) in enumerate(tss.split(dias), start=1):
    pos_ent = np.flatnonzero(fechas_ent.isin(dias[d_ent]).to_numpy())
    pos_val = np.flatnonzero(fechas_ent.isin(dias[d_val]).to_numpy())
    esquemas.append(("temporal", k, pos_ent, pos_val))

kf = KFold(n_splits=N_PLIEGUES, shuffle=True, random_state=42)
for k, (pos_ent, pos_val) in enumerate(kf.split(X_ent), start=1):
    esquemas.append(("al_azar", k, pos_ent, pos_val))

for esquema, k, pos_ent, pos_val in esquemas:
    f_val = fechas_ent.iloc[pos_val]
    print(f"{esquema:<9} pliegue {k}: entrena {len(pos_ent):>7,} filas | valida {len(pos_val):>7,} filas "
          f"({f_val.min().date()} a {f_val.max().date()})")

temporal  pliegue 1: entrena 426,860 filas | valida 121,960 filas (2016-05-04 a 2016-05-07)
temporal  pliegue 2: entrena 548,820 filas | valida 121,960 filas (2016-05-08 a 2016-05-11)
temporal  pliegue 3: entrena 670,780 filas | valida 121,960 filas (2016-05-12 a 2016-05-15)
al_azar   pliegue 1: entrena 528,493 filas | valida 264,247 filas (2016-04-20 a 2016-05-15)
al_azar   pliegue 2: entrena 528,493 filas | valida 264,247 filas (2016-04-20 a 2016-05-15)
al_azar   pliegue 3: entrena 528,494 filas | valida 264,246 filas (2016-04-20 a 2016-05-15)


## 3. Modelos

Las mismas configuraciones de la sesión pasada. El naive estacional ("igual que hace 7 días") se agrega solo como referencia: no se entrena, así que no tarda nada.

In [5]:
MODELOS = {
    "Random Forest sin regularizar (final)": lambda: RandomForestRegressor(
        n_estimators=50, max_samples=0.3, n_jobs=-1, random_state=42),
    "Red neuronal (segundo)": lambda: MLPRegressor(
        hidden_layer_sizes=(64, 32), batch_size=1024, random_state=42),
    "Naive estacional (referencia)": lambda: NaiveEstacional(),
}


def nuevo_pipeline(fabrica):
    modelo = fabrica()
    preparacion = "passthrough" if isinstance(modelo, NaiveEstacional) else nueva_preparacion()
    return Pipeline([("preparacion", preparacion), ("modelo", modelo)])

## 4. Validación cruzada

In [6]:
PARCIAL = "pliegues_parcial.csv"
filas = pd.read_csv(PARCIAL).to_dict("records") if os.path.exists(PARCIAL) else []
hechos = {(f["esquema"], f["pliegue"], f["modelo"]) for f in filas}

for esquema, k, pos_ent, pos_val in esquemas:
    for nombre, fabrica in MODELOS.items():
        if (esquema, k, nombre) in hechos:
            continue                                  # ya calculado en una corrida anterior
        X_a, y_a = X_ent.iloc[pos_ent], y_ent.iloc[pos_ent]
        X_v, y_v = X_ent.iloc[pos_val], y_ent.iloc[pos_val]
        f_val = fechas_ent.iloc[pos_val]

        pipe = nuevo_pipeline(fabrica)
        inicio = time.perf_counter()
        pipe.fit(X_a, y_a)
        segundos = time.perf_counter() - inicio
        pred = pipe.predict(X_v)
        filas.append({
            "modelo": nombre, "esquema": esquema, "pliegue": k,
            "valida_desde": str(f_val.min().date()), "valida_hasta": str(f_val.max().date()),
            "filas_entrenamiento": len(pos_ent), "filas_validacion": len(pos_val),
            "wape": wape(y_v, pred),
            "costo_relativo": costo_relativo(y_v, pred, X_v["SELL_PRICE"]),
            "sesgo": sesgo(y_v, pred),
            "segundos_entrenamiento": segundos,
        })
        print(f"{esquema:<9} pliegue {k} | {nombre:<38} WAPE {filas[-1]['wape']:.4f} | "
              f"costo {filas[-1]['costo_relativo']:.4f} | sesgo {filas[-1]['sesgo']:+.3f} | {segundos:.0f} s")

        pd.DataFrame(filas).to_csv(PARCIAL, index=False)   # guarda el avance
        del pipe, pred, X_a, y_a, X_v, y_v                  # libera memoria
        gc.collect()

pliegues = pd.DataFrame(filas)

al_azar   pliegue 1 | Random Forest sin regularizar (final)  WAPE 0.8402 | costo 0.5377 | sesgo +0.023 | 30 s
al_azar   pliegue 1 | Red neuronal (segundo)                 WAPE 0.8377 | costo 0.5483 | sesgo -0.013 | 301 s
al_azar   pliegue 1 | Naive estacional (referencia)          WAPE 0.8536 | costo 0.5627 | sesgo -0.036 | 0 s
al_azar   pliegue 2 | Random Forest sin regularizar (final)  WAPE 0.8426 | costo 0.5320 | sesgo +0.040 | 29 s
al_azar   pliegue 2 | Red neuronal (segundo)                 WAPE 0.8233 | costo 0.5675 | sesgo -0.062 | 265 s
al_azar   pliegue 2 | Naive estacional (referencia)          WAPE 0.8547 | costo 0.5583 | sesgo -0.025 | 0 s
al_azar   pliegue 3 | Random Forest sin regularizar (final)  WAPE 0.8404 | costo 0.5366 | sesgo +0.023 | 29 s
al_azar   pliegue 3 | Red neuronal (segundo)                 WAPE 0.8390 | costo 0.5433 | sesgo +0.007 | 247 s
al_azar   pliegue 3 | Naive estacional (referencia)          WAPE 0.8543 | costo 0.5642 | sesgo -0.036 | 0 s


## 5. Tabla: cada pliegue, promedio y desviación

In [7]:
METRICAS = ["wape", "costo_relativo", "sesgo", "segundos_entrenamiento"]

resumen = (pliegues.groupby(["modelo", "esquema"])[METRICAS]
           .agg(["mean", "std"]))
resumen.columns = [f"{m}_{e}" for m, e in resumen.columns]

promedio = pliegues.groupby(["modelo", "esquema"], as_index=False)[METRICAS].mean().assign(pliegue="promedio")
desviacion = pliegues.groupby(["modelo", "esquema"], as_index=False)[METRICAS].std().assign(pliegue="desviacion")
tabla = pd.concat([pliegues, promedio, desviacion], ignore_index=True)
tabla["pliegue"] = tabla["pliegue"].astype(str)
tabla = tabla.sort_values(["esquema", "modelo"], kind="stable")
tabla.to_csv("validacion_proyecto.csv", index=False)
print("Guardado validacion_proyecto.csv")

resumen.round(4)

Guardado validacion_proyecto.csv


wape_mean  wape_std  \
modelo                                esquema                         
Naive estacional (referencia)         al_azar      0.8542    0.0006   
                                      temporal     0.8483    0.0367   
Random Forest sin regularizar (final) al_azar      0.8411    0.0013   
                                      temporal     0.8414    0.0322   
Red neuronal (segundo)                al_azar      0.8333    0.0087   
                                      temporal     0.9863    0.1053   

                                                costo_relativo_mean  \
modelo                                esquema                         
Naive estacional (referencia)         al_azar                0.5617   
                                      temporal               0.5633   
Random Forest sin regularizar (final) al_azar                0.5354   
                                      temporal               0.5393   
Red neuronal (segundo)                al_azar                0.5530   
                                      temporal               0.5512   

                                                costo_relativo_std  \
modelo                                esquema                        
Naive estacional (referencia)         al_azar               0.0030   
                                      temporal              0.0140   
Random Forest sin regularizar (final) al_azar               0.0030   
                                      temporal              0.0149   
Red neuronal (segundo)                al_azar               0.0128   
                                      temporal              0.0189   

                                                sesgo_mean  sesgo_std  \
modelo                                esquema                           
Naive estacional (referencia)         al_azar      -0.0322     0.0064   
                                      temporal     -0.0485     0.0744   
Random Forest sin regularizar (final) al_azar       0.0289     0.0098   
                                      temporal      0.0081     0.0954   
Red neuronal (segundo)                al_azar      -0.0230     0.0353   
                                      temporal      0.2031     0.1024   

                                                segundos_entrenamiento_mean  \
modelo                                esquema                                 
Naive estacional (referencia)         al_azar                        0.0003   
                                      temporal                       0.0003   
Random Forest sin regularizar (final) al_azar                       29.3484   
                                      temporal                      21.7165   
Red neuronal (segundo)                al_azar                      271.0581   
                                      temporal                     212.2275   

                                                segundos_entrenamiento_std  
modelo                                esquema                               
Naive estacional (referencia)         al_azar                       0.0000  
                                      temporal                      0.0000  
Random Forest sin regularizar (final) al_azar                       0.4453  
                                      temporal                      8.1780  
Red neuronal (segundo)                al_azar                      27.8924  
                                      temporal                     59.5598

## 6. Comparaciones para las respuestas

In [8]:
def fila(modelo, esquema):
    return resumen.loc[(modelo, esquema)]

FINAL, SEGUNDO = "Random Forest sin regularizar (final)", "Red neuronal (segundo)"

for esquema in ["temporal", "al_azar"]:
    f, s = fila(FINAL, esquema), fila(SEGUNDO, esquema)
    print(f"\n== Esquema {esquema} ==")
    for metrica in ["wape", "costo_relativo"]:
        dif = s[f"{metrica}_mean"] - f[f"{metrica}_mean"]
        print(f"{metrica:<15} final {f[f'{metrica}_mean']:.4f} ± {f[f'{metrica}_std']:.4f} | "
              f"segundo {s[f'{metrica}_mean']:.4f} ± {s[f'{metrica}_std']:.4f} | "
              f"diferencia (segundo - final) {dif:+.4f}")

# Diferencia pliegue a pliegue: los dos modelos se evaluan en los mismos pliegues,
# asi que se puede ver si el final gana en todos o solo en promedio.
for esquema in ["temporal", "al_azar"]:
    p = pliegues[pliegues["esquema"] == esquema].pivot(index="pliegue", columns="modelo", values="wape")
    gana = (p[FINAL] < p[SEGUNDO]).sum()
    print(f"\n{esquema}: el final tiene menor WAPE en {gana} de {len(p)} pliegues")
    print((p[SEGUNDO] - p[FINAL]).round(4).rename("WAPE segundo - final").to_string())


== Esquema temporal ==
wape            final 0.8414 ± 0.0322 | segundo 0.9863 ± 0.1053 | diferencia (segundo - final) +0.1450
costo_relativo  final 0.5393 ± 0.0149 | segundo 0.5512 ± 0.0189 | diferencia (segundo - final) +0.0119

== Esquema al_azar ==
wape            final 0.8411 ± 0.0013 | segundo 0.8333 ± 0.0087 | diferencia (segundo - final) -0.0077
costo_relativo  final 0.5354 ± 0.0030 | segundo 0.5530 ± 0.0128 | diferencia (segundo - final) +0.0176

temporal: el final tiene menor WAPE en 3 de 3 pliegues
pliegue
1    0.302
2    0.084
3    0.049

al_azar: el final tiene menor WAPE en 0 de 3 pliegues
pliegue
1   -0.0025
2   -0.0193
3   -0.0014
